# Кандидатогенерация для поиска услуг (Avito)

Решение задачи отбора топ-50 кандидатов по поисковому запросу.
Архитектура: гибридный поиск (TF-IDF + rubert-tiny2) с учетом категории и локации.

In [ ]:
import time
import numpy as np
import pandas as pd
import polars as pl
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# проверка устройства
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'устройство: {device}')

In [ ]:
# загрузка данных бенчмарка
benchmark_queries = pd.read_parquet('dataset/benchmark_queries.parquet')
benchmark_items = pd.read_parquet('dataset/benchmark_items.parquet')

print(f'запросов: {len(benchmark_queries)}, товаров: {len(benchmark_items)}')

In [ ]:
# склейка текстовых признаков через polars для экономии памяти
pl_items = pl.from_pandas(benchmark_items)
t = pl_items['item_title_raw'].fill_null('') + ' '
p = pl_items['item_infm_params_text'].fill_null('') + ' '
d = pl_items['item_description_raw'].fill_null('')

# заголовок повторяется 3 раза для повышения веса
# буква ё заменяется на е для унификации
benchmark_items['corpus_text'] = (
    (t + t + t + p + p + d)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

pl_queries = pl.from_pandas(benchmark_queries)
q = pl_queries['search_query'].fill_null('') + ' '
qp = pl_queries['search_infm_params_text'].fill_null('')

benchmark_queries['query_text'] = (
    (q + qp)
    .str.to_lowercase()
    .str.replace_all('ё', 'е')
    .to_pandas()
)

In [ ]:
# обучение tf-idf с биграммами и логарифмической частотой слова
bench_vectorizer = TfidfVectorizer(
    min_df=2,
    sublinear_tf=True,
    ngram_range=(1, 2)
)
X_bench_corpus = bench_vectorizer.fit_transform(benchmark_items['corpus_text'])
X_bench_queries = bench_vectorizer.transform(benchmark_queries['query_text'])

print(f'признаков в словаре: {X_bench_corpus.shape[1]}')

In [ ]:
# загрузка модели rubert-tiny2 и получение плотных эмбеддингов
model = SentenceTransformer('cointegrated/rubert-tiny2', device=device)
model.max_seq_length = 128

import os
if os.path.exists('bench_query_embeddings.npy') and os.path.exists('bench_corpus_embeddings.npy'):
    print('загрузка сохраненных эмбеддингов с диска')
    bench_query_embeddings = np.load('bench_query_embeddings.npy')
    bench_corpus_embeddings = np.load('bench_corpus_embeddings.npy')
else:
    print('кодирование запросов и корпуса')
    bench_query_embeddings = model.encode(
        benchmark_queries['query_text'].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    bench_corpus_embeddings = model.encode(
        benchmark_items['corpus_text'].tolist(),
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save('bench_query_embeddings.npy', bench_query_embeddings)
    np.save('bench_corpus_embeddings.npy', bench_corpus_embeddings)

print(f'размерность эмбеддингов: {bench_corpus_embeddings.shape}')

In [ ]:
# подготовка метаданных для фильтрации и ранжирования
bench_corpus_cats = benchmark_items['item_category_id'].values
bench_corpus_locs = benchmark_items['item_location_id'].values
bench_corpus_ids = benchmark_items['item_id'].values

bench_query_cats = benchmark_queries['search_category'].values
bench_query_locs = benchmark_queries['search_location_id'].values
bench_query_ids = benchmark_queries['query_id'].values

total_queries = len(benchmark_queries)
batch_size = 100
predictions = []

# поиск кандидатов батчами
for start_idx in range(0, total_queries, batch_size):
    end_idx = min(start_idx + batch_size, total_queries)
    
    s_tfidf = (X_bench_queries[start_idx:end_idx] @ X_bench_corpus.T).toarray()
    s_dense = bench_query_embeddings[start_idx:end_idx] @ bench_corpus_embeddings.T
    
    b_cats = bench_query_cats[start_idx:end_idx, None]
    b_locs = bench_query_locs[start_idx:end_idx, None]
    
    # категория 0 означает сквозной поиск по всем разделам
    cat_match = (b_cats == 0) | (b_cats == bench_corpus_cats[None, :])
    loc_match = (b_locs == bench_corpus_locs[None, :])
    
    # локация дает множитель 3.2, чужая категория штрафуется
    multiplier = np.where(cat_match, np.where(loc_match, 3.2, 1.0), 0.001)
    
    s_tfidf_b = s_tfidf * multiplier
    s_dense_b = s_dense * multiplier
    
    max_t = np.maximum(s_tfidf_b.max(axis=1, keepdims=True), 1e-6)
    max_d = np.maximum(s_dense_b.max(axis=1, keepdims=True), 1e-6)
    
    # смешивание скоров: 40% лексика и 60% семантика
    s_hybrid = 0.40 * (s_tfidf_b / max_t) + 0.60 * (s_dense_b / max_d)
    
    # отбор топ-50 кандидатов
    top50_idx = np.argpartition(-s_hybrid, 50, axis=1)[:, :50]
    
    for i in range(end_idx - start_idx):
        items = bench_corpus_ids[top50_idx[i]]
        predictions.append(' '.join(items))

print(f'сгенерировано предсказаний: {len(predictions)}')

In [ ]:
# формирование файла ответа и проверки формата
answer_df = pd.DataFrame({
    'query_id': bench_query_ids,
    'answer': predictions
})

assert len(answer_df) == 2452, 'неверное число строк'
assert answer_df['answer'].isna().sum() == 0, 'есть пропуски в ответах'
assert all(len(ans.split(' ')) == 50 for ans in answer_df['answer']), 'в ответе не 50 товаров'
assert answer_df['query_id'].nunique() == 2452, 'есть дубликаты query_id'

answer_df.to_csv('answer.csv', index=False)
print('файл answer.csv сохранен')
print(answer_df.head(2))